In [1]:
pip install pandas textblob googletrans==4.0.0-rc1

Note: you may need to restart the kernel to use updated packages.


In [2]:
!pip install pyspellchecker

In [3]:
!pip install emoji

In [4]:
%pip install -U pip setuptools wheel
%pip install scikit-learn numpy scipy matplotlib

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [5]:
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('vader_lexicon')   # required for SentimentIntensityAnalyzer


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\rajin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\rajin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\rajin\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\rajin\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\rajin\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [6]:
from nltk.stem.wordnet import WordNetLemmatizer
from nltk.tag import pos_tag
from nltk.tokenize import word_tokenize
from nltk.tokenize import sent_tokenize
from textblob import TextBlob
from nltk.corpus import stopwords
import re,string
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from bs4 import BeautifulSoup
from spellchecker import SpellChecker
import emoji
import pandas as pd
from googletrans import Translator
from sklearn import datasets, linear_model
from sklearn.model_selection import train_test_split
from matplotlib import pyplot as plt
from nltk.classify.scikitlearn import SklearnClassifier
from sklearn.naive_bayes import MultinomialNB,BernoulliNB
from sklearn.linear_model import LogisticRegression,SGDClassifier
from sklearn.svm import SVC
from sklearn.metrics import f1_score
from nltk.corpus import wordnet
from sklearn.metrics import classification_report

#google translate
translator = Translator(service_urls =['translate.google.com'])

#pyspellchecker
spell = SpellChecker()

In [7]:
#label 0 means negative and 4 means positive
data = pd.read_csv("test_data.csv",skip_blank_lines=True,encoding = "latin")
data

,ï»¿Label,number,date,no_query,name,Tweet
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."
...,...,...,...,...,...,...
39996,4,1960186342,Fri May 29 07:33:44 PDT 2009,NO_QUERY,Madelinedugganx,My GrandMa is making Dinenr with my Mum
39997,4,1960186409,Fri May 29 07:33:43 PDT 2009,NO_QUERY,OffRoad_Dude,Mid-morning snack time... A bowl of cheese noo...
39998,4,1960186429,Fri May 29 07:33:44 PDT 2009,NO_QUERY,Falchion,@ShaDeLa same here say it like from the Termi...
39999,4,1960186445,Fri May 29 07:33:44 PDT 2009,NO_QUERY,jonasobsessedx,@DestinyHope92 im great thaanks wbuu?


In [8]:
l=[]
for i in data["ï»¿Label"]:
    if(i==0):
        l.append("negative")
    else:
        l.append("positive")
data['label']=l

In [9]:
#dropping unwanted columns
data=data.drop(columns=['number', 'date','name','no_query','ï»¿Label'])

In [10]:
#removes all emojis
def deEmojify(inputString):
    return inputString.encode('ascii', 'ignore').decode('ascii')

In [11]:
def contractions():
    return {
        "ain't":"is not",
        "amn't":"am not",
        "aren't":"are not",
        "can't":"cannot",
        "'cause":"because",
        "couldn't":"could not",
        "couldn't've":"could not have",
        "could've":"could have",
        "daren't":"dare not",
        "daresn't":"dare not",
        "dasn't":"dare not",
        "didn't":"did not",
        "doesn't":"does not",
        "don't":"do not",
        "e'er":"ever",
        "em":"them",
        "everyone's":"everyone is",
        "finna":"fixing to",
        "gimme":"give me",
        "gonna":"going to",
        "gon't":"go not",
        "gotta":"got to",
        "hadn't":"had not",
        "hasn't":"has not",
        "haven't":"have not",
        "he'd":"he would",
        "he'll":"he will",
        "he's":"he is",
        "he've":"he have",
        "how'd":"how would",
        "how'll":"how will",
        "how're":"how are",
        "how's":"how is",
        "ily":"I love you",
        "Ily":"I love you",
        "Ihy":"I hate you",
        "ihy":"I hate you",
        "imy":"I miss you",
        "Imy":"I miss you",
        "I'd":"I would",
        "I'll":"I will",
        "I'm":"I am",
        "im":"I am",
        "I'm'a":"I am about to",
        "I'm'o":"I am going to",
        "isn't":"is not",
        "it'd":"it would",
        "it'll":"it will",
        "it's":"it is",
        "I've":"I have",
        "kinda":"kind of",
        "let's":"let us",
        "mayn't":"may not",
        "may've":"may have",
        "mightn't":"might not",
        "might've":"might have",
        "mustn't":"must not",
        "mustn't've":"must not have",
        "must've":"must have",
        "needn't":"need not",
        "ne'er":"never",
        "o'":"of",
        "o'er":"over",
        "ol'":"old",
        "oughtn't":"ought not",
        "shalln't":"shall not",
        "shan't":"shall not",
        "she'd":"she would",
        "she'll":"she will",
        "she's":"she is",
        "shouldn't":"should not",
        "shouldn't've":"should not have",
        "should've":"should have",
        "somebody's":"somebody is",
        "someone's":"someone is",
        "something's":"something is",
        "that'd":"that would",
        "that'll":"that will",
        "that're":"that are",
        "that's":"that is",
        "there'd":"there would",
        "there'll":"there will",
        "there're":"there are",
        "there's":"there is",
        "these're":"these are",
        "they'd":"they would",
        "they'll":"they will",
        "they're":"they are",
        "they've":"they have",
        "this's":"this is",
        "those're":"those are",
        "'tis":"it is",
        "'twas":"it was",
        "wanna":"want to",
        "wasn't":"was not",
        "we'd":"we would",
        "we'd've":"we would have",
        "we'll":"we will",
        "we're":"we are",
        "weren't":"were not",
        "we've":"we have",
        "what'd":"what did",
        "what'll":"what will",
        "what're":"what are",
        "what's":"what is",
        "what've":"what have",
        "when's":"when is",
        "where'd":"where did",
        "where're":"where are",
        "where's":"where is",
        "where've":"where have",
        "which's":"which is",
        "who'd":"who would",
        "who'd've":"who would have",
        "who'll":"who will",
        "who're":"who are",
        "who's":"who is",
        "who've":"who have",
        "why'd":"why did",
        "why're":"why are",
        "why's":"why is",
        "won't":"will not",
        "wouldn't":"would not",
        "would've":"would have",
        "y'all":"you all",
        "you'd":"you would",
        "you'll":"you will",
        "you're":"you are",
        "you've":"you have",
        "Whatcha":"What are you",
        "luv":"love",
        "sux":"sucks",
        "shit":"bad",
        "tmr":"tomorrow",
        "tmrw":"tomorrow",
        "u":"you",
        "ur":"your",
        "k":"okay",
        "ok":"okay",
        "da":"the",
        "tom":"tomorrow",
        "Tom":"tomorrow",
        "v'll":"we will",
        "wassup":"what is up with you",
        "waddup":"what is up with you",
        "yo":"greet",
        "hey":"greet",
        "lol":"laugh",
        "lmao":"laugh",
        "Lmao":"laugh",
        "rofl":"laugh",
        "y":"why",
        "wut":"what",
        "wat":"what",
        "stfu":"angry",
        "wtf":"angry",
        "ya":"yes",
        "yeah":"yes",
        "ummmm":"confused",
        "ummm":"confused",
        "umm":"confused",
        "hmmm":"confused",
        "i'm":"I am",
        "awww":"amazement",
        "Awww":"amazement",
        "aww":"amazement",
        "Aww":"amazement",
        "can't":"cannot",
        "Can't":"cannot",
        "CAN'T":"cannot",
        "awe":"amazement",
        "Awe":"amazement",
        "ugh":"sad",
        "ughh":"sad",
        "Ugh":"sad",
        "Ughh":"sad",
        "UGHH":"sad",
        "ughhhh":"sad",
        "ughhh":"sad"
        }

In [13]:
def emoticons():

    return {
        ":)":"smiley",
        ":‑)":"smiley",
        ":-]":"smiley",
        ":-3":"smiley",
        ":->":"smiley",
        "8-)":"smiley",
        ":-}":"smiley",
        ":)":"smiley",
        ":]":"smiley",
        ":3":"smiley",
        ":>":"smiley",
        "8)":"smiley",
        ":}":"smiley",
        ":o)":"smiley",
        ":c)":"smiley",
        ":^)":"smiley",
        "=]":"smiley",
        "=)":"smiley",
        ":-))":"smiley",
        ":‑D":"smiley",
        "8‑D":"smiley",
        "x‑D":"smiley",
        "X‑D":"smiley",
        ":D":"smiley",
        "8D":"smiley",
        "xD":"smiley",
        "XD":"smiley",
        ":‑(":"sad",
        ":‑c":"sad",
        ":‑<":"sad",
        ":‑[":"sad",
        ":(":"sad",
        ":c":"sad",
        ":<":"sad",
        ":[":"sad",
        ":-||":"sad",
        ">:[":"sad",
        ":{":"sad",
        ":@":"sad",
        ">:(":"sad",
        ":'‑(":"sad",
        ":'(":"sad",
        ":((((":"sad",
        ":(((":"sad",
        ":((":"sad",
        ":(":"sad",
        ":/":"sad",
        ":///":"sad",
        ":////":"sad",
        "://///":"sad",
        "://":"sad",
        ":///////":"sad",
        ":////":"sad",
        "-_-":"angry",
        ":|":"normal",
        ";)":"playful",
        ";D":"playful",
        ":‑P":"playful",
        "X‑P":"playful",
        "x‑p":"playful",
        ":‑p":"playful",
        ":‑Þ":"playful",
        ":‑þ":"playful",
        ":‑b":"playful",
        ":P":"playful",
        "XP":"playful",
        "xp":"playful",
        ":p":"playful",
        ":Þ":"playful",
        ":þ":"playful",
        ":b":"playful",
        "<3":"love"
    }

In [20]:
def lemmatization(sent):
    lemmatize=WordNetLemmatizer()
    sentence_after_lemmatization=[]
    for word,tag in pos_tag(word_tokenize(sent)):
        if(tag[0:2]=="NN"):
            pos='n'
        elif(tag[0:2]=="VB"):
            pos='v'
        else:
            pos='a'
        lem=lemmatize.lemmatize(word,pos)
        sentence_after_lemmatization.append(lem)
    st=""
    for i in sentence_after_lemmatization:
        if(i!="be" and i!="is" and len(i)!=1):
            st=st+" "+i
    #print("lemi",st)
    c=0
    list_text=st.split()
    flag=0
    new_st=""
    for i in list_text:
        temp=i
        if(flag==1):
            flag=0
            continue
        if(i=="not" and (c+1)<len(list_text)):
            for syn in wordnet.synsets(list_text[c+1]):
                antonyms=[]
                for l in syn.lemmas():
                    #print(l)
                    if l.antonyms():
                        antonyms.append(l.antonyms()[0].name())
                        #print(antonyms)
                        temp=antonyms[0]
                        flag=1
                        break
                if(flag==1):
                    break
        new_st=new_st+" "+temp
        c+=1
    #print(new_st)
    return new_st

In [15]:
def removal_of_noise(sent):
    clean_sent = []
    temp_st = ""
    list_sent = sent.split(" ")
    c = 0
    d = contractions()
    emoji = emoticons()

    for word in list_sent:
        # Removal of url
        word = re.sub(r"http\S+", "", word)
        word = re.sub(r"[www.][a-zA-Z0-9_]+[.com]", "", word)

        # Removal of account handles '@'
        word = re.sub("(@[A-Za-z0-9_]+)", "", word)

        # Replacing emoticons with their respective words
        if word in emoji.keys():
            word = emoji[word]

        # Replacing short form words with their full form
        if word.lower() in d.keys():
            word = d[word.lower()]

        if c == 0:
            temp_st = word
        else:
            temp_st = temp_st + " " + word
        c = c + 1

    sent = temp_st
    stop_words = set(stopwords.words('english'))
    stop_words.add('is')
    stop_words.remove('not')

    for word in word_tokenize(sent):
        if (word.lower() not in stop_words and
            word.lower() not in string.punctuation and
            word != "'" and word != '"'):

            # FIX: Store correction result first
            corrected = spell.correction(word.lower())

            # Check if spell.correction() returned None
            if corrected is None:
                word = word.lower()
            else:
                word = corrected

            # Only proceed if word exists
            if word:
                word = re.sub("[0-9]+", "", word)
                word = re.sub("[.]+", " ", word)
                word = re.sub("[-]+", " ", word)
                word = re.sub("[_]+", " ", word)
                word = re.sub("~", " ", word)
                word = word.strip()

                if word and len(word) != 1:
                    clean_sent.append(word.lower())

    cleaned_st = ""
    for i in clean_sent:
        cleaned_st = cleaned_st + " " + i

    return lemmatization(cleaned_st)

In [16]:
#nltk module to get the sentiment polarity
def sentiment_analysis(sent):
        sid = SentimentIntensityAnalyzer()
        #print("-------------------------------------")
        print(sent)
        #print("-------------------------------------")
        ss = sid.polarity_scores(sent)
        x=ss['pos']
        y=ss['neg']
        print(x-y)
        print("-------------------------------------")
        return x-y

In [17]:
def start(text):
    #removes html tags
    text =BeautifulSoup(text).get_text()
    text =text.replace("’","'")
    new_text=sent_tokenize(text)
    #print((new_text))
    result=0
    new_str=""
    #removing emojis
    for i in new_text:
        j=deEmojify(i)
        res=removal_of_noise(j)
        new_str=new_str+" "+res
    return new_str

In [18]:
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\rajin\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [19]:
def start(text):
    # Add debugging to print out the input text
    print("Input text:", text)

    # Check if the input is a string
    if not isinstance(text, str):
        print("Input text is not a string.")
        return ""

    #removes html tags
    text = BeautifulSoup(text).get_text()
    text = text.replace("’", "'")
    new_text = sent_tokenize(text)
    result = 0
    new_str = ""

    # Add more debugging to print out intermediate steps
    print("After removing HTML tags:", text)

    #removing emojis
    for i in new_text:
        j = deEmojify(i)
        res = removal_of_noise(j)
        new_str = new_str + " " + res

    print("After removing emojis and noise:", new_str)

    # Return the preprocessed text as 'x'
    x = new_str
    return x

In [21]:
# Read from file cleaned tweets and store in a cleaned tweets column in the DataFrame
filename = "cleaned_tweet.txt"
with open(filename) as f:
    lines = f.read().splitlines()

# Check the length of lines read from the file
print("Number of lines read from file:", len(lines))

# Check the length of the DataFrame data
print("Number of rows in DataFrame data:", len(data))

# If the lengths match, assign the lines to the "cleaned_tweets" column
if len(lines) == len(data):
    data["cleaned_tweets"] = lines
    print("Successfully assigned values to 'cleaned_tweets' column.")
else:
    print("Lengths do not match. Investigate further.")

# Print the first few rows of the DataFrame to verify the new column
print(data.head())

Number of lines read from file: 0
Number of rows in DataFrame data: 40001
Lengths do not match. Investigate further.
                                               Tweet     label
0  @switchfoot http://twitpic.com/2y1zl - Awww, t...  negative
1  is upset that he can't update his Facebook by ...  negative
2  @Kenichan I dived many times for the ball. Man...  negative
3    my whole body feels itchy and like its on fire   negative
4  @nationwideclass no, it's not behaving at all....  negative


In [22]:
# Example: Load and clean your tweets
tweets = ["tweet 1 text", "tweet 2 text", "tweet 3 text"]  # Your raw data

# Clean the tweets
clean_list = []
for tweet in tweets:
    cleaned_tweet = tweet.strip().lower()  # Your cleaning logic
    clean_list.append(cleaned_tweet)

# Or if you already have cleaned data:
# clean_list = your_cleaned_data

In [23]:
# --------------------------------------------
# 1) Twitter/Social Media Adjectives List
# --------------------------------------------

adjectives_list = [
    # POSITIVE EMOTIONS
    "good", "great", "best", "better", "awesome", "amazing", "wonderful",
    "perfect", "excellent", "beautiful", "lovely", "nice", "happy", "excited",
    "glad", "pleased", "thrilled", "delighted", "blessed", "grateful", "thankful",
    "proud", "lucky", "fortunate", "positive", "optimistic", "hopeful", "inspired",
    "fantastic", "fabulous", "incredible", "outstanding", "spectacular", "brilliant",
    "terrific", "superb", "marvelous", "phenomenal", "impressive", "stunning",
    "gorgeous", "pretty", "cute", "adorable", "sweet", "charming",
    "cool", "fun", "funny", "hilarious", "entertaining", "interesting", "exciting",

    # NEGATIVE EMOTIONS
    "bad", "worst", "worse", "terrible", "awful", "horrible", "disgusting",
    "sad", "unhappy", "depressed", "miserable", "upset", "angry", "mad", "furious",
    "annoyed", "frustrated", "irritated", "disappointed", "hurt", "broken",
    "sick", "tired", "exhausted", "bored", "annoying", "stupid", "dumb", "crazy",
    "wrong", "fake", "false", "lame", "useless", "pathetic", "ridiculous",
    "nasty", "gross", "ugly", "mean", "rude", "evil", "hateful", "toxic",
    "negative", "pessimistic", "hopeless", "scared", "afraid", "worried", "nervous",
    "anxious", "stressed", "confused", "lost", "embarrassed", "ashamed",

    # INTENSITY WORDS
    "very", "so", "too", "much", "more", "most", "less", "least", "super",
    "really", "totally", "completely", "absolutely", "extremely", "highly",
    "quite", "pretty", "somewhat", "fairly", "rather", "bit", "little",
    "big", "huge", "massive", "enormous", "giant", "large", "small", "tiny",
    "full", "empty", "half", "whole", "entire", "complete",

    # QUALITY
    "right", "correct", "true", "real", "actual", "genuine", "legit", "authentic",
    "serious", "important", "major", "main", "key", "critical", "vital", "essential",
    "necessary", "urgent", "clear", "obvious", "sure", "certain", "definite",
    "simple", "easy", "hard", "difficult", "complicated", "complex", "tough",
    "quick", "fast", "slow", "late", "early", "ready", "busy", "free",

    # SOCIAL
    "new", "old", "latest", "recent", "current", "modern", "fresh", "viral",
    "trending", "hot", "popular", "famous", "top", "favorite", "classic",
    "live", "online", "digital", "virtual", "social", "public", "private",
    "official", "unofficial", "original", "unique", "special", "rare",
    "common", "normal", "typical", "usual", "regular", "average", "basic",
    "random", "weird", "strange", "odd", "unusual", "different", "same",

    # PERSONAL
    "own", "personal", "my", "our", "your", "their", "his", "her",
    "single", "married", "young", "old", "alone", "lonely", "together",

    # TECH & DIGITAL
    "smart", "dumb", "broken", "fixed", "updated", "outdated", "buggy",
    "fast", "slow", "laggy", "smooth", "glitchy", "online", "offline",

    # TIME & FREQUENCY
    "daily", "weekly", "monthly", "yearly", "never", "always", "sometimes",
    "often", "rarely", "frequent", "constant", "temporary", "permanent",
    "short", "long", "brief", "quick", "instant", "immediate",
]

# Remove duplicates + sort
adjectives_list = sorted(list(set(adjectives_list)))

# Extra negative keyword
negative = ["not"]


# --------------------------------------------
# 2) Extract All Adjective Words From Tweets
# --------------------------------------------

from nltk.tokenize import word_tokenize

all_words = []

for tweet in data["Tweet"]:
    for word in word_tokenize(tweet.lower()):   # lowercase for matching
        if word in adjectives_list or word in negative:
            all_words.append(word)

print("Total adjective words found:", len(all_words))
print(all_words[:50])  # first 50 for preview



Total adjective words found: 49448
['upset', 'his', 'my', 'whole', 'not', 'mad', 'not', 'whole', 'long', 'bit', 'bit', 'my', 'never', 'not', 'really', 'hurt', 'not', 'always', 'most', 'so', 'much', 'my', 'her', 'not', 'sad', 'so', 'mad', 'depressed', 'my', 'new', 'not', 'too', 'my', 'sick', 'too', 'sick', 'not', 'good', 'more', 'my', 'really', 'my', 'my', 'sad', 'sad', 'sad', 'sad', 'sad', 'my', 'happy']


In [24]:
import nltk
nltk.download('punkt_tab')
nltk.download('punkt')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\rajin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\rajin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [25]:
print(data.columns)

Index(['Tweet', 'label'], dtype='object')


In [26]:
#creating a frequency distribution of each adjectives.
import nltk
BagOfWords = nltk.FreqDist(all_words)
BagOfWords
len(BagOfWords)

259

In [27]:
# listing the  5000 most frequent words
word_features = list(BagOfWords.keys())[:5000]
len(word_features)
#word_features

259

In [28]:
#after preprocessing data
data

,Tweet,label
0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",negative
1,is upset that he can't update his Facebook by ...,negative
2,@Kenichan I dived many times for the ball. Man...,negative
3,my whole body feels itchy and like its on fire,negative
4,"@nationwideclass no, it's not behaving at all....",negative
...,...,...
39996,My GrandMa is making Dinenr with my Mum,positive
39997,Mid-morning snack time... A bowl of cheese noo...,positive
39998,@ShaDeLa same here say it like from the Termi...,positive
39999,@DestinyHope92 im great thaanks wbuu?,positive


In [29]:
#assigning feature for each row in clean_tweets
new_list=[]
for i in data["Tweet"]:
    st=""
    for j in i.split():
        if(j in word_features):
            st=st+" "+j
    new_list.append(st)

data["cleaned_tweets"]=new_list

In [30]:
data

,Tweet,label,cleaned_tweets
0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",negative,
1,is upset that he can't update his Facebook by ...,negative,upset his
2,@Kenichan I dived many times for the ball. Man...,negative,
3,my whole body feels itchy and like its on fire,negative,my whole
4,"@nationwideclass no, it's not behaving at all....",negative,not
...,...,...,...
39996,My GrandMa is making Dinenr with my Mum,positive,my
39997,Mid-morning snack time... A bowl of cheese noo...,positive,
39998,@ShaDeLa same here say it like from the Termi...,positive,same
39999,@DestinyHope92 im great thaanks wbuu?,positive,great


In [31]:
#Spliting into test data and train data
y=data["label"]
x=data.drop('label',axis=1)
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.15,random_state=42)

In [32]:
x_train.shape

(34000, 2)

In [33]:
import pandas as pd

# Ensure x_train and x_test are DataFrames
assert isinstance(x_train, pd.DataFrame), "x_train is not a DataFrame"
assert isinstance(x_test, pd.DataFrame), "x_test is not a DataFrame"

# Initialize X_train and X_test
X_train = pd.DataFrame(columns=['Tweet', 'cleaned_tweets'])
X_test = pd.DataFrame(columns=['Tweet', 'cleaned_tweets'])

# Append x_train and x_test to X_train and X_test
X_train = pd.concat([X_train, x_train], ignore_index=True)
X_test = pd.concat([X_test, x_test], ignore_index=True)

# Append labels to Y_train and Y_test
Y_train = y_train.copy()
Y_test = y_test.copy()

print("X_train:", X_train.head())
print("X_test:", X_test.head())
print("Y_train:", Y_train[:5])
print("Y_test:", Y_test[:5])

X_train:                                                Tweet cleaned_tweets
0                school resumes this monday. gaaah.                
1  @annakimx3 i feel u dawgggg. i finished my pap...             my
2  I think I need to get laid. Sad revolution I h...               
3  is finally home after a long day... but no com...      long good
4               im so proud to be out of the closet        so proud
X_test:                                                Tweet cleaned_tweets
0    ahahah i tried to do the bob the builder dance                
1             @britneyspears sorry i missed youBrit                
2                                  will &amp; grace                
3  UGH......I took medicine on an empty stomach  ...          empty
4              @eglinski Thank you for not doing so          not so
Y_train: 10422    negative
12358    negative
1377     negative
20254    positive
38600    positive
Name: label, dtype: object
Y_test: 32824    positive
16298    negati

In [34]:
# Splitting into train sets for training
training_set = []
for tweet, label in zip(X_train["cleaned_tweets"], Y_train):
    training_set.append((tweet.split(), label))

def list_to_dict(words_list):
    return dict([(word, True) for word in words_list])

# Format training set
training_set_formatted = [(list_to_dict(element[0]), element[1]) for element in training_set]

# Print or use the formatted training set
print(training_set_formatted)

[({}, 'negative'), ({'my': True}, 'negative'), ({}, 'negative'), ({'long': True, 'good': True}, 'positive'), ({'so': True, 'proud': True}, 'positive'), ({'really': True}, 'positive'), ({'more': True, 'my': True}, 'negative'), ({}, 'positive'), ({}, 'negative'), ({'right': True}, 'negative'), ({}, 'positive'), ({'very': True, 'glad': True}, 'negative'), ({'our': True, 'always': True}, 'positive'), ({}, 'negative'), ({'my': True}, 'negative'), ({'good': True}, 'negative'), ({'amazing': True, 'always': True, 'short': True, 'my': True, 'so': True, 'early': True}, 'negative'), ({'too': True}, 'positive'), ({'sad': True}, 'negative'), ({'long': True, 'your': True, 'tired': True, 'my': True}, 'negative'), ({'big': True}, 'negative'), ({'complete': True}, 'negative'), ({}, 'negative'), ({}, 'negative'), ({'my': True, 'really': True, 'afraid': True}, 'negative'), ({'really': True}, 'positive'), ({}, 'positive'), ({'more': True}, 'negative'), ({'your': True}, 'negative'), ({}, 'negative'), ({'so

In [35]:
# Splitting into test sets for testing
test_set = []
for tweet, label in zip(X_test["cleaned_tweets"], Y_test):
    test_set.append((tweet.split(), label))

def list_to_dict(words_list):
    return dict([(word, True) for word in words_list])

# Format test set
test_set_formatted = [(list_to_dict(element[0]), element[1]) for element in test_set]

# Print or use the formatted test set
print(test_set_formatted)

[({}, 'positive'), ({}, 'negative'), ({}, 'positive'), ({'empty': True}, 'negative'), ({'not': True, 'so': True}, 'positive'), ({'nice': True, 'big': True, 'sweet': True}, 'positive'), ({}, 'negative'), ({'little': True}, 'positive'), ({}, 'negative'), ({}, 'positive'), ({}, 'positive'), ({}, 'negative'), ({}, 'negative'), ({}, 'negative'), ({}, 'positive'), ({}, 'negative'), ({'my': True, 'more': True}, 'negative'), ({}, 'negative'), ({'big': True, 'so': True, 'pretty': True, 'your': True}, 'positive'), ({}, 'positive'), ({'right': True, 'more': True}, 'negative'), ({'my': True}, 'negative'), ({'so': True}, 'positive'), ({}, 'positive'), ({'my': True}, 'negative'), ({}, 'positive'), ({'positive': True, 'my': True, 'personal': True}, 'positive'), ({'my': True, 'good': True}, 'positive'), ({'not': True}, 'positive'), ({'excited': True, 'her': True}, 'positive'), ({}, 'positive'), ({}, 'positive'), ({'your': True}, 'positive'), ({'whole': True}, 'positive'), ({'my': True}, 'positive'), (

In [36]:
from sklearn.metrics import recall_score,precision_score
#making a list of classifiers with their names
classifiers=[]
#making a list of classifiers with their accuracy
accuracy=[]

In [37]:
#naive bayes classifier
classifier = nltk.NaiveBayesClassifier.train(training_set_formatted)

ground_truth = [r[1] for r in test_set_formatted]
preds = [classifier.classify(r[0]) for r in test_set_formatted]
f1score=f1_score(ground_truth, preds, labels = ['negative', 'positive'], average = 'micro')
recallscore=recall_score(ground_truth, preds, labels = ['negative', 'positive'], average = 'macro')
precisionscore=precision_score(ground_truth, preds, labels = ['negative', 'positive'], average = 'weighted')

#accuracy
print("Original Naive Bayes Algo accuracy percent:", (nltk.classify.accuracy(classifier, test_set_formatted))*100)
# print("F1-score : ",100*f1score)
# print("Recall Score : ",100*recallscore)
# print("Precision Score : ",100*precisionscore)
print()

classifier.show_most_informative_features(15)
            
classifiers.append([classifier,"naive bayes classifier"])

accuracy.append([(nltk.classify.accuracy(classifier, test_set_formatted))*100,"NB"])

print("Original Naive Bayes\n")
target_names = [ 'positive','negative']
print(classification_report(Y_test, preds, target_names=target_names))

Original Naive Bayes Algo accuracy percent: 59.79003499416764

Most Informative Features
                    sick = True           negati : positi =     21.2 : 1.0
                   worst = True           negati : positi =     17.4 : 1.0
                     sad = True           negati : positi =     14.6 : 1.0
               depressed = True           negati : positi =     11.0 : 1.0
                positive = True           positi : negati =     11.0 : 1.0
                terrible = True           negati : positi =     10.2 : 1.0
                  lonely = True           negati : positi =      9.7 : 1.0
                  broken = True           negati : positi =      9.5 : 1.0
            disappointed = True           negati : positi =      9.4 : 1.0
                   upset = True           negati : positi =      9.2 : 1.0
               hilarious = True           positi : negati =      9.0 : 1.0
                   nasty = True           negati : positi =      8.4 : 1.0
           

In [38]:
#Multinomail naive bayes
MNB_clf = SklearnClassifier(MultinomialNB())
MNB_clf.train(training_set_formatted)
print("Multinomail naive bayes classifier accuracy percent:", (nltk.classify.accuracy(MNB_clf, test_set_formatted))*100)

ground_truth = [r[1] for r in test_set_formatted]
preds = [MNB_clf.classify(r[0]) for r in test_set_formatted]
f1score=f1_score(ground_truth, preds, labels = ['negative', 'positive'], average = 'micro')
recallscore=recall_score(ground_truth, preds, labels = ['negative', 'positive'], average = 'macro')
precisionscore=precision_score(ground_truth, preds, labels = ['negative', 'positive'], average = 'weighted')

# print("F1-score : ",100*f1score)
# print("Recall Score : ",100*recallscore)
# print("Precision Score : ",100*precisionscore)

accuracy.append([(nltk.classify.accuracy(MNB_clf, test_set_formatted))*100,"MNB"])

classifiers.append([MNB_clf,"Multinomail naive bayes classifier"])

print("Multinomail naive bayes\n")
target_names = [ 'positive','negative']
print(classification_report(Y_test, preds, target_names=target_names))

Multinomail naive bayes classifier accuracy percent: 59.40676553907682
Multinomail naive bayes

              precision    recall  f1-score   support

    positive       0.65      0.42      0.51      3022
    negative       0.57      0.77      0.65      2979

    accuracy                           0.59      6001
   macro avg       0.61      0.60      0.58      6001
weighted avg       0.61      0.59      0.58      6001



In [39]:
#Bernouli naive bayes
BNB_clf = SklearnClassifier(BernoulliNB())
BNB_clf.train(training_set_formatted)
print("Bernoulli naive bayes classifier accuracy percent:", (nltk.classify.accuracy(BNB_clf, test_set_formatted))*100)

ground_truth = [r[1] for r in test_set_formatted]
preds = [BNB_clf.classify(r[0]) for r in test_set_formatted]
f1score=f1_score(ground_truth, preds, labels = ['negative', 'positive'], average = 'micro')
recallscore=recall_score(ground_truth, preds, labels = ['negative', 'positive'], average = 'macro')
precisionscore=precision_score(ground_truth, preds, labels = ['negative', 'positive'], average = 'weighted')

# print("F1-score : ",100*f1score)
# print("Recall Score : ",100*recallscore)
# print("Precision Score : ",100*precisionscore)

accuracy.append([(nltk.classify.accuracy(BNB_clf, test_set_formatted))*100,"BNB"])

classifiers.append([BNB_clf,"Bernouli classifier"])

print("Bernouli naive bayes\n")
target_names = [ 'positive','negative']
print(classification_report(Y_test, preds, target_names=target_names))

Bernoulli naive bayes classifier accuracy percent: 59.55674054324279
Bernouli naive bayes

              precision    recall  f1-score   support

    positive       0.65      0.44      0.52      3022
    negative       0.57      0.76      0.65      2979

    accuracy                           0.60      6001
   macro avg       0.61      0.60      0.59      6001
weighted avg       0.61      0.60      0.58      6001



In [40]:
#Logistic regression
LogReg_clf = SklearnClassifier(LogisticRegression())
LogReg_clf.train(training_set_formatted)
print("Logistic Regression classifier accuracy percent:", (nltk.classify.accuracy(LogReg_clf, test_set_formatted))*100)

ground_truth = [r[1] for r in test_set_formatted]
preds = [LogReg_clf.classify(r[0]) for r in test_set_formatted]
f1score=f1_score(ground_truth, preds, labels = ['negative', 'positive'], average = 'micro')
recallscore=recall_score(ground_truth, preds, labels = ['negative', 'positive'], average = 'macro')
precisionscore=precision_score(ground_truth, preds, labels = ['negative', 'positive'], average = 'weighted')

# print("F1-score : ",100*f1score)
# print("Recall Score : ",100*recallscore)
# print("Precision Score : ",100*precisionscore)

accuracy.append([(nltk.classify.accuracy(LogReg_clf, test_set_formatted))*100,"LogReg"])


classifiers.append([LogReg_clf,"Bernouli LogisticRegression_classifier"])

print("Logistic regression\n")
target_names = [ 'positive','negative']
print(classification_report(Y_test, preds, target_names=target_names))

Logistic Regression classifier accuracy percent: 59.606732211298116
Logistic regression

              precision    recall  f1-score   support

    positive       0.65      0.44      0.52      3022
    negative       0.57      0.76      0.65      2979

    accuracy                           0.60      6001
   macro avg       0.61      0.60      0.59      6001
weighted avg       0.61      0.60      0.59      6001



In [41]:
#Stochastic Gradient Descent classifier
SGD_clf = SklearnClassifier(SGDClassifier())
SGD_clf.train(training_set_formatted)
print("Stochastic Gradient Descent Classifier_classifier accuracy percent:", (nltk.classify.accuracy(SGD_clf, test_set_formatted))*100)

ground_truth = [r[1] for r in test_set_formatted]
preds = [SGD_clf.classify(r[0]) for r in test_set_formatted]
f1score=f1_score(ground_truth, preds, labels = ['negative', 'positive'], average = 'micro')
recallscore=recall_score(ground_truth, preds, labels = ['negative', 'positive'], average = 'macro')
precisionscore=precision_score(ground_truth, preds, labels = ['negative', 'positive'], average = 'weighted')

# print("F1-score : ",100*f1score)
# print("Recall Score : ",100*recallscore)
# print("Precision Score : ",100*precisionscore)

accuracy.append([(nltk.classify.accuracy(SGD_clf, test_set_formatted))*100,"SGD"])


classifiers.append([SGD_clf,"SGD classifier"])

print("Stochastic Gradient Descent\n")
target_names = [ 'positive','negative']
print(classification_report(Y_test, preds, target_names=target_names))

Stochastic Gradient Descent Classifier_classifier accuracy percent: 58.606898850191634
Stochastic Gradient Descent

              precision    recall  f1-score   support

    positive       0.66      0.38      0.48      3022
    negative       0.56      0.80      0.66      2979

    accuracy                           0.59      6001
   macro avg       0.61      0.59      0.57      6001
weighted avg       0.61      0.59      0.57      6001



In [42]:
#Support vector classifier
SVC_clf = SklearnClassifier(SVC())
SVC_clf.train(training_set_formatted)
print("Support vector classifier accuracy percent:", (nltk.classify.accuracy(SVC_clf, test_set_formatted))*100)

ground_truth = [r[1] for r in test_set_formatted]
preds = [SVC_clf.classify(r[0]) for r in test_set_formatted]
f1score=f1_score(ground_truth, preds, labels = ['negative', 'positive'], average = 'micro')
recallscore=recall_score(ground_truth, preds, labels = ['negative', 'positive'], average = 'macro')
precisionscore=precision_score(ground_truth, preds, labels = ['negative', 'positive'], average = 'weighted')

# print("F1-score : ",100*f1score)
# print("Recall Score : ",100*recallscore)
# print("Precision Score : ",100*precisionscore)

accuracy.append([(nltk.classify.accuracy(SVC_clf, test_set_formatted))*100,"SVC"])

classifiers.append([SVC_clf,"SVC classifier"])

print("Support vector classifier\n")
target_names = [ 'positive','negative']
print(classification_report(Y_test, preds, target_names=target_names))

Support vector classifier accuracy percent: 59.356773871021495
Support vector classifier

              precision    recall  f1-score   support

    positive       0.64      0.44      0.52      3022
    negative       0.57      0.75      0.65      2979

    accuracy                           0.59      6001
   macro avg       0.60      0.59      0.58      6001
weighted avg       0.60      0.59      0.58      6001



In [43]:
#Max Entropy classifier
from nltk.classify import  MaxentClassifier
from sklearn.metrics import f1_score, accuracy_score
from sklearn.metrics import f1_score

def max_ent(training_set_formatted):
    numIterations = 100
    algorithm = nltk.classify.MaxentClassifier.ALGORITHMS[0]
    classifier = nltk.MaxentClassifier.train(training_set_formatted, algorithm, max_iter=numIterations)
    classifier.show_most_informative_features(10)
    return classifier

maxent_classifier=max_ent(training_set_formatted)


ground_truth = [r[1] for r in test_set_formatted]

preds = [maxent_classifier.classify(r[0]) for r in test_set_formatted]

f1score=f1_score(ground_truth, preds, labels = ['negative', 'positive'], average = 'micro')
recallscore=recall_score(ground_truth, preds, labels = ['negative', 'positive'], average = 'macro')
precisionscore=precision_score(ground_truth, preds, labels = ['negative', 'positive'], average = 'weighted')

print("Accuracy : ",f1_score(ground_truth, preds, labels = ['positive', 'negative'], average = 'micro')*100)
# print("F1-score : ",100*f1score)
# print("Recall Score : ",100*recallscore)
# print("Precision Score : ",100*precisionscore)

accuracy.append([(nltk.classify.accuracy(maxent_classifier, test_set_formatted))*100,"MaxEnt"])



classifiers.append([maxent_classifier,"Max Entropy classifier"])

print("Max Entropy\n")
target_names = [ 'positive','negative']
print(classification_report(Y_test, preds, target_names=target_names))

  ==> Training (100 iterations)

      Iteration    Log Likelihood    Accuracy
      ---------------------------------------
             1          -0.69315        0.501
             2          -0.69266        0.608
             3          -0.69218        0.608
             4          -0.69170        0.608
             5          -0.69123        0.608
             6          -0.69075        0.608
             7          -0.69028        0.608
             8          -0.68981        0.608
             9          -0.68934        0.608
            10          -0.68888        0.608
            11          -0.68842        0.608
            12          -0.68796        0.608
            13          -0.68750        0.608
            14          -0.68704        0.608
            15          -0.68659        0.608
            16          -0.68614        0.608
            17          -0.68569        0.608
            18          -0.68525        0.608
            19          -0.68480        0.608
 

In [44]:
from nltk.classify import ClassifierI
from statistics import mode

# Defininig the ensemble model class

class EnsembleClassifier(ClassifierI):

    def __init__(self, *classifiers):
        self._classifiers = classifiers

    # returns the classification based on majority of votes
    def classify(self, features):
        votes = []
        for c in self._classifiers:
            v = c.classify(features)
            votes.append(v)
        return mode(votes)

In [45]:
def hybrid(test_set_formatted):
    ensemble_clf = EnsembleClassifier(classifiers[0][0], classifiers[1][0], classifiers[2][0], classifiers[3][0],classifiers[4][0],classifiers[5][0],classifiers[6][0])

    # List of only feature dictionary from the featureset list of tuples
    feature_list = [f[0] for f in test_set_formatted]
    global c
    # Looping over each to classify each review
    ensemble_preds = [ensemble_clf.classify(features) for features in feature_list]

    # Initialize counter for correct predictions
    c = 0

    for i in range(len(ensemble_preds)):
        if ensemble_preds[i] == test_set_formatted[i][1]:  # Accessing label from test_set_formatted
            c += 1

    return ensemble_preds

# Call the hybrid function
c = 0
preds = hybrid(test_set_formatted)

# Extract ground truth labels from test_set_formatted
ground_truth = [r[1] for r in test_set_formatted]

# Calculate accuracy
accuracy = 100 * c / len(preds)
print("Accuracy of hybrid: ", accuracy)

# Print classification report
target_names = ['positive', 'negative']
print(classification_report(ground_truth, preds, target_names=target_names))

Accuracy of hybrid:  59.54007665389102
              precision    recall  f1-score   support

    positive       0.64      0.44      0.52      3022
    negative       0.57      0.75      0.65      2979

    accuracy                           0.60      6001
   macro avg       0.61      0.60      0.59      6001
weighted avg       0.61      0.60      0.59      6001



In [46]:
accuracy

59.54007665389102

In [47]:
def features(text):
    new_list=[]
    for i in text.split():
        if(i in adjectives_list):
            new_list.append(i)
    return new_list

In [48]:
def text_classify(text):
    cleaned_text=start(text)
    temp=features(cleaned_text)
    test_data=list_to_dict(temp)
    print(temp)
    print("Tweet given by user : ",text)
    for i in classifiers:
        print(i[1])
        determined_label=i[0].classify(test_data)
        print("This Tweet is ",determined_label)
        print("------------------------------")
    c=0
    print("Hybrid model")
    testset_data=[]
    testset_data.append([test_data,""])
    lab=hybrid(testset_data)
    print("This Tweet is ",lab[0])

In [49]:
nltk.download('twitter_samples')

[nltk_data] Downloading package twitter_samples to
[nltk_data]     C:\Users\rajin\AppData\Roaming\nltk_data...
[nltk_data]   Package twitter_samples is already up-to-date!


True

In [50]:
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\rajin\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [51]:
from nltk.corpus import twitter_samples

positive_tweets = twitter_samples.strings('positive_tweets.json')
negative_tweets = twitter_samples.strings('negative_tweets.json')
x=negative_tweets[:1000]
st=""
for i in x:
    st=st+" "+i
st

' hopeless for tmr :( Everything in the kids section of IKEA is so cute. Shame I\'m nearly 19 in 2 months :( @Hegelbon That heart sliding into the waste basket. :( “@ketchBurning: I hate Japanese call him "bani" :( :(”\n\nMe too Dang starting next week I have "work" :( oh god, my babies\' faces :( https://t.co/9fcwGvaki0 @RileyMcDonough make me smile :(( @f0ggstar @stuartthull work neighbour on motors. Asked why and he said hates the updates on search :( http://t.co/XvmTUikWln why?:("@tahuodyy: sialan:( https://t.co/Hv1i0xcrL2" Athabasca glacier was there in #1948 :-( #athabasca #glacier #jasper #jaspernationalpark #alberta #explorealberta #… http://t.co/dZZdqmf7Cz I have a really good m&amp;g idea but I\'m never going to meet them :((( @Rampageinthebox mare ivan :( @SophiaMascardo happy trip, keep safe. see you soon :* :( I\'m so tired hahahah :( @GrumpyCockney With knee replacements they get you up &amp; about the same day. :-(   Ouch. relate to the "sweet n\' sour" kind of "bi-polar

In [52]:
from spellchecker import SpellChecker

# Initialize spell checker
spell = SpellChecker()

In [53]:
import nltk

# Download all required NLTK resources
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\rajin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\rajin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\rajin\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\rajin\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\rajin\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\rajin\AppData\Roaming\nltk_dat

True

In [54]:
text_classify(st)

Input text:  hopeless for tmr :( Everything in the kids section of IKEA is so cute. Shame I'm nearly 19 in 2 months :( @Hegelbon That heart sliding into the waste basket. :( “@ketchBurning: I hate Japanese call him "bani" :( :(”

Me too Dang starting next week I have "work" :( oh god, my babies' faces :( https://t.co/9fcwGvaki0 @RileyMcDonough make me smile :(( @f0ggstar @stuartthull work neighbour on motors. Asked why and he said hates the updates on search :( http://t.co/XvmTUikWln why?:("@tahuodyy: sialan:( https://t.co/Hv1i0xcrL2" Athabasca glacier was there in #1948 :-( #athabasca #glacier #jasper #jaspernationalpark #alberta #explorealberta #… http://t.co/dZZdqmf7Cz I have a really good m&amp;g idea but I'm never going to meet them :((( @Rampageinthebox mare ivan :( @SophiaMascardo happy trip, keep safe. see you soon :* :( I'm so tired hahahah :( @GrumpyCockney With knee replacements they get you up &amp; about the same day. :-(   Ouch. relate to the "sweet n' sour" kind of "bi-p

In [55]:
from googletrans import Translator

In [56]:
#input from the user which will be used to classify
def hinglish(input_text):
    translator = Translator(service_urls=['translate.google.co.in'])
    x=translator.translate(input_text,src="hi",dest="en")
    text_classify(x.text)

In [57]:
from textblob import TextBlob

def hinglish2(input_text):
    l = input_text.split()
    st = ""
    for i in l:
        word_obj = TextBlob(i)
        try:
            lang = word_obj.detect_language()
        except Exception as e:
            lang = 'en'  # default to English if detection fails
        if lang == "hi":
            translator = Translator(service_urls=['translate.google.co.in'])
            x = translator.translate(i, src="hi", dest="en")
            st = st + " " + x.text
        else:
            st = st + " " + i
    text_classify(st)


In [58]:
def func(input_text):
    l=input_text.split()
    flag=0
    for i in l:
        k=len(i)
        if(k<3):
            flag=1
            hinglish(input_text)
    if(not(flag)):
        hinglish2(input_text)

In [59]:
func("tum bekar ho :( ")

Input text: you are useless :(
After removing HTML tags: you are useless :(
After removing emojis and noise:   useless sad
['useless', 'sad']
Tweet given by user :  you are useless :(
naive bayes classifier
This Tweet is  negative
------------------------------
Multinomail naive bayes classifier
This Tweet is  negative
------------------------------
Bernouli classifier
This Tweet is  negative
------------------------------
Bernouli LogisticRegression_classifier
This Tweet is  negative
------------------------------
SGD classifier
This Tweet is  negative
------------------------------
SVC classifier
This Tweet is  negative
------------------------------
Max Entropy classifier
This Tweet is  negative
------------------------------
Hybrid model
This Tweet is  negative
Input text: you are useless :(
After removing HTML tags: you are useless :(
After removing emojis and noise:   useless sad
['useless', 'sad']
Tweet given by user :  you are useless :(
naive bayes classifier
This Tweet is  ne

In [58]:
func("tu diwani hai")

Input text: you are crazy
After removing HTML tags: you are crazy
After removing emojis and noise:   crazy
['crazy']
Tweet given by user :  you are crazy
naive bayes classifier
This Tweet is  positive
------------------------------
Multinomail naive bayes classifier
This Tweet is  positive
------------------------------
Bernouli classifier
This Tweet is  positive
------------------------------
Bernouli LogisticRegression_classifier
This Tweet is  positive
------------------------------
SGD classifier
This Tweet is  positive
------------------------------
SVC classifier
This Tweet is  positive
------------------------------
Max Entropy classifier
This Tweet is  positive
------------------------------
Hybrid model
This Tweet is  positive


In [59]:
func("tum kharab ho")

Input text: you are bad
After removing HTML tags: you are bad
After removing emojis and noise:   bad
['bad']
Tweet given by user :  you are bad
naive bayes classifier
This Tweet is  negative
------------------------------
Multinomail naive bayes classifier
This Tweet is  negative
------------------------------
Bernouli classifier
This Tweet is  negative
------------------------------
Bernouli LogisticRegression_classifier
This Tweet is  negative
------------------------------
SGD classifier
This Tweet is  negative
------------------------------
SVC classifier
This Tweet is  negative
------------------------------
Max Entropy classifier
This Tweet is  negative
------------------------------
Hybrid model
This Tweet is  negative


In [61]:
func("arrey waah! I'm very proud of you")

Input text: wow wow!I'm very proud of you
After removing HTML tags: wow wow!I'm very proud of you
After removing emojis and noise:   wow wow proud
['proud']
Tweet given by user :  wow wow!I'm very proud of you
naive bayes classifier
This Tweet is  positive
------------------------------
Multinomail naive bayes classifier
This Tweet is  positive
------------------------------
Bernouli classifier
This Tweet is  positive
------------------------------
Bernouli LogisticRegression_classifier
This Tweet is  positive
------------------------------
SGD classifier
This Tweet is  positive
------------------------------
SVC classifier
This Tweet is  positive
------------------------------
Max Entropy classifier
This Tweet is  positive
------------------------------
Hybrid model
This Tweet is  positive


In [63]:
func("tum ache nahi ho")

Input text: you are not good
After removing HTML tags: you are not good
After removing emojis and noise:   evil
['evil']
Tweet given by user :  you are not good
naive bayes classifier
This Tweet is  negative
------------------------------
Multinomail naive bayes classifier
This Tweet is  negative
------------------------------
Bernouli classifier
This Tweet is  negative
------------------------------
Bernouli LogisticRegression_classifier
This Tweet is  negative
------------------------------
SGD classifier
This Tweet is  positive
------------------------------
SVC classifier
This Tweet is  negative
------------------------------
Max Entropy classifier
This Tweet is  negative
------------------------------
Hybrid model
This Tweet is  negative


In [64]:
func("tum stupid ho")

Input text: you are stupid
After removing HTML tags: you are stupid
After removing emojis and noise:   stupid
['stupid']
Tweet given by user :  you are stupid
naive bayes classifier
This Tweet is  negative
------------------------------
Multinomail naive bayes classifier
This Tweet is  negative
------------------------------
Bernouli classifier
This Tweet is  negative
------------------------------
Bernouli LogisticRegression_classifier
This Tweet is  negative
------------------------------
SGD classifier
This Tweet is  negative
------------------------------
SVC classifier
This Tweet is  negative
------------------------------
Max Entropy classifier
This Tweet is  negative
------------------------------
Hybrid model
This Tweet is  negative


In [145]:
func("Aaj ka weather bohot fresh aur lovely hai.")

Input text: Today's weather is very fresh and lovely.
After removing HTML tags: Today's weather is very fresh and lovely.
After removing emojis and noise:   today weather fresh lovely
['fresh', 'lovely']
Tweet given by user :  Today's weather is very fresh and lovely.
naive bayes classifier
This Tweet is  positive
------------------------------
Multinomail naive bayes classifier
This Tweet is  positive
------------------------------
Bernouli classifier
This Tweet is  positive
------------------------------
Bernouli LogisticRegression_classifier
This Tweet is  positive
------------------------------
SGD classifier
This Tweet is  positive
------------------------------
SGD classifier
This Tweet is  positive
------------------------------
SVC classifier
This Tweet is  positive
------------------------------
SGD classifier
This Tweet is  positive
------------------------------
SVC classifier
This Tweet is  positive
------------------------------
Max Entropy classifier
This Tweet is  positi

In [65]:
func("isko dekh ke muje gussa ara hai")

Input text: I feel angry after seeing this
After removing HTML tags: I feel angry after seeing this
After removing emojis and noise:   feel angry seeing
['angry']
Tweet given by user :  I feel angry after seeing this
naive bayes classifier
This Tweet is  negative
------------------------------
Multinomail naive bayes classifier
This Tweet is  negative
------------------------------
Bernouli classifier
This Tweet is  negative
------------------------------
Bernouli LogisticRegression_classifier
This Tweet is  negative
------------------------------
SGD classifier
This Tweet is  positive
------------------------------
SVC classifier
This Tweet is  negative
------------------------------
Max Entropy classifier
This Tweet is  negative
------------------------------
Hybrid model
This Tweet is  negative


In [66]:
func("tum cute ho")

Input text: U cute
After removing HTML tags: U cute
After removing emojis and noise:   cute
['cute']
Tweet given by user :  U cute
naive bayes classifier
This Tweet is  positive
------------------------------
Multinomail naive bayes classifier
This Tweet is  positive
------------------------------
Bernouli classifier
This Tweet is  positive
------------------------------
Bernouli LogisticRegression_classifier
This Tweet is  positive
------------------------------
SGD classifier
This Tweet is  positive
------------------------------
SVC classifier
This Tweet is  positive
------------------------------
Max Entropy classifier
This Tweet is  positive
------------------------------
Hybrid model
This Tweet is  positive


In [69]:
func("Timeline completely messed up hai, experience horrible ho gaya")

Input text: Timeline is completely messed up, experience has been horrible
After removing HTML tags: Timeline is completely messed up, experience has been horrible
After removing emojis and noise:   timeline completely mess experience horrible
['completely', 'horrible']
Tweet given by user :  Timeline is completely messed up, experience has been horrible
naive bayes classifier
This Tweet is  negative
------------------------------
Multinomail naive bayes classifier
This Tweet is  negative
------------------------------
Bernouli classifier
This Tweet is  negative
------------------------------
Bernouli LogisticRegression_classifier
This Tweet is  negative
------------------------------
SGD classifier
This Tweet is  negative
------------------------------
SVC classifier
This Tweet is  negative
------------------------------
Max Entropy classifier
This Tweet is  negative
------------------------------
Hybrid model
This Tweet is  negative
Input text: Timeline is completely messed up, exper

In [146]:
func("App slow, buggy aur unreliable ho chuka :(")

Input text: The app has become slow, buggy and ready to be thrown :(
After removing HTML tags: The app has become slow, buggy and ready to be thrown :(
After removing emojis and noise:   ape become slow buggy ready thrown sad
['slow', 'buggy', 'ready', 'sad']
Tweet given by user :  The app has become slow, buggy and ready to be thrown :(
naive bayes classifier
This Tweet is  negative
------------------------------
Multinomail naive bayes classifier
This Tweet is  negative
------------------------------
Bernouli classifier
This Tweet is  negative
------------------------------
Bernouli LogisticRegression_classifier
This Tweet is  negative
------------------------------
SGD classifier
This Tweet is  negative
------------------------------
SGD classifier
This Tweet is  negative
------------------------------
SVC classifier
This Tweet is  negative
------------------------------
SGD classifier
This Tweet is  negative
------------------------------
SVC classifier
This Tweet is  negative
----

In [140]:
func("Aaj ka din bohot bad aur depressing gaya :-<.")

Input text: Today has been a very bad and depressing day :-<.
After removing HTML tags: Today has been a very bad and depressing day :-<.
After removing emojis and noise:   today bad depressing day
['bad']
Tweet given by user :  Today has been a very bad and depressing day :-<.
naive bayes classifier
This Tweet is  negative
------------------------------
Multinomail naive bayes classifier
This Tweet is  negative
------------------------------
Bernouli classifier
This Tweet is  negative
------------------------------
Bernouli LogisticRegression_classifier
This Tweet is  negative
------------------------------
SGD classifier
This Tweet is  negative
------------------------------
SGD classifier
This Tweet is  negative
------------------------------
SVC classifier
This Tweet is  negative
------------------------------
SGD classifier
This Tweet is  negative
------------------------------
SVC classifier
This Tweet is  negative
------------------------------
Max Entropy classifier
This Tweet 

In [142]:
func("Aaj ka din really good aur productive gaya. Feeling very happy!")

Input text: Today was a really good and productive day.Feeling very happy!
After removing HTML tags: Today was a really good and productive day.Feeling very happy!
After removing emojis and noise:   today really good productive day feel happy
['really', 'good', 'happy']
Tweet given by user :  Today was a really good and productive day.Feeling very happy!
naive bayes classifier
This Tweet is  positive
------------------------------
Multinomail naive bayes classifier
This Tweet is  positive
------------------------------
Bernouli classifier
This Tweet is  positive
------------------------------
Bernouli LogisticRegression_classifier
This Tweet is  positive
------------------------------
SGD classifier
This Tweet is  positive
------------------------------
SGD classifier
This Tweet is  positive
------------------------------
SVC classifier
This Tweet is  positive
------------------------------
SGD classifier
This Tweet is  positive
------------------------------
SVC classifier
This Tweet 

In [147]:
func("Bhai tu full shaitaan mood mein hai :P")

Input text: Brother, you are in full devil mood :P
After removing HTML tags: Brother, you are in full devil mood :P
After removing emojis and noise:   brother full devil mood playful
['full']
Tweet given by user :  Brother, you are in full devil mood :P
naive bayes classifier
This Tweet is  positive
------------------------------
Multinomail naive bayes classifier
This Tweet is  positive
------------------------------
Bernouli classifier
This Tweet is  positive
------------------------------
Bernouli LogisticRegression_classifier
This Tweet is  positive
------------------------------
SGD classifier
This Tweet is  positive
------------------------------
SGD classifier
This Tweet is  positive
------------------------------
SVC classifier
This Tweet is  positive
------------------------------
SGD classifier
This Tweet is  positive
------------------------------
SVC classifier
This Tweet is  positive
------------------------------
Max Entropy classifier
This Tweet is  positive
------------

In [86]:
func("Yeh sab bakwaas hai, kuch samajh nahi aata")

Input text:  Yeh sab bakwaas hai, kuch samajh nahi aata
After removing HTML tags:  Yeh sab bakwaas hai, kuch samajh nahi aata
After removing emojis and noise:   yet sab bakwaas have much smash nazi data
['much']
Tweet given by user :   Yeh sab bakwaas hai, kuch samajh nahi aata
naive bayes classifier
This Tweet is  negative
------------------------------
Multinomail naive bayes classifier
This Tweet is  negative
------------------------------
Bernouli classifier
This Tweet is  negative
------------------------------
Bernouli LogisticRegression_classifier
This Tweet is  negative
------------------------------
SGD classifier
This Tweet is  positive
------------------------------
SVC classifier
This Tweet is  negative
------------------------------
Max Entropy classifier
This Tweet is  negative
------------------------------
Hybrid model
This Tweet is  negative


In [87]:
func("Ladai karna achi baat nahi hai")

Input text:  Ladai karna achi baat nahi hai
After removing HTML tags:  Ladai karna achi baat nahi hai
After removing emojis and noise:   lanai karma ache beat nazi have
[]
Tweet given by user :   Ladai karna achi baat nahi hai
naive bayes classifier
This Tweet is  positive
------------------------------
Multinomail naive bayes classifier
This Tweet is  positive
------------------------------
Bernouli classifier
This Tweet is  positive
------------------------------
Bernouli LogisticRegression_classifier
This Tweet is  positive
------------------------------
SGD classifier
This Tweet is  positive
------------------------------
SVC classifier
This Tweet is  positive
------------------------------
Max Entropy classifier
This Tweet is  positive
------------------------------
Hybrid model
This Tweet is  positive


In [88]:
func("Main bahut pareshan hoon aaj kal")

Input text:  Main bahut pareshan hoon aaj kal
After removing HTML tags:  Main bahut pareshan hoon aaj kal
After removing emojis and noise:   main baht parisian soon raj pal
['main']
Tweet given by user :   Main bahut pareshan hoon aaj kal
naive bayes classifier
This Tweet is  negative
------------------------------
Multinomail naive bayes classifier
This Tweet is  negative
------------------------------
Bernouli classifier
This Tweet is  negative
------------------------------
Bernouli LogisticRegression_classifier
This Tweet is  negative
------------------------------
SGD classifier
This Tweet is  negative
------------------------------
SVC classifier
This Tweet is  negative
------------------------------
Max Entropy classifier
This Tweet is  negative
------------------------------
Hybrid model
This Tweet is  negative


In [62]:
func("is app ke slow hone se muje frustation ho rhi hai")

Input text: I am getting frustrated with this app being slow.
After removing HTML tags: I am getting frustrated with this app being slow.
After removing emojis and noise:   get frustrate ape slow
['slow']
Tweet given by user :  I am getting frustrated with this app being slow.
naive bayes classifier
This Tweet is  negative
------------------------------
Multinomail naive bayes classifier
This Tweet is  negative
------------------------------
Bernouli classifier
This Tweet is  negative
------------------------------
Bernouli LogisticRegression_classifier
This Tweet is  negative
------------------------------
SGD classifier
This Tweet is  negative
------------------------------
SVC classifier
This Tweet is  negative
------------------------------
Max Entropy classifier
This Tweet is  negative
------------------------------
Hybrid model
This Tweet is  negative
Input text: I am getting frustrated with this app being slow.
After removing HTML tags: I am getting frustrated with this app bein

In [57]:
func("aaj maine kapde khareede , par voh acha nahi hai")

Input text: Today I bought clothes, but they are not good
After removing HTML tags: Today I bought clothes, but they are not good
After removing emojis and noise:   today buy clothes evil
['evil']
Tweet given by user :  Today I bought clothes, but they are not good
naive bayes classifier
This Tweet is  negative
------------------------------
Multinomail naive bayes classifier
This Tweet is  negative
------------------------------
Bernouli classifier
This Tweet is  negative
------------------------------
Bernouli LogisticRegression_classifier
This Tweet is  negative
------------------------------
SGD classifier
This Tweet is  negative
------------------------------
SVC classifier
This Tweet is  negative
------------------------------
Max Entropy classifier
This Tweet is  negative
------------------------------
Hybrid model
This Tweet is  negative


In [62]:
func("do you think I like this?")

Input text: do you think i like this?
After removing HTML tags: do you think i like this?
After removing emojis and noise:   think like
[]
Tweet given by user :  do you think i like this?
naive bayes classifier
This Tweet is  positive
------------------------------
Multinomail naive bayes classifier
This Tweet is  positive
------------------------------
Bernouli classifier
This Tweet is  positive
------------------------------
Bernouli LogisticRegression_classifier
This Tweet is  positive
------------------------------
SGD classifier
This Tweet is  positive
------------------------------
SVC classifier
This Tweet is  positive
------------------------------
Max Entropy classifier
This Tweet is  positive
------------------------------
Hybrid model
This Tweet is  positive
Input text: do you think i like this?
After removing HTML tags: do you think i like this?
After removing emojis and noise:   think like
[]
Tweet given by user :  do you think i like this?
naive bayes classifier
This Twee